# 01 Data and folds

Builds the common population of site-days that every method is scored on, and the eighteen station-held-out folds with the manifest that records which stations each fold may learn from. Every later notebook reads its files.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the counterfactual bridge method of the `pynrpf` package. **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes, and a *slot* counts intervals from midnight (slot 24 is 06:00).

**Inputs.** `dataset/final/dataset_alpha.parquet` and `dataset_beta.parquet` (hashes verified against `config/evaluation.yaml`).

**Outputs.** `results/01_data_folds/`: `site_days_alpha.parquet` and `site_days_beta.parquet` (one row per station-day: completeness, confidence, labelled window), `population.csv`, `fold_manifest.csv`; `results/manifests/01_data_folds.json`.

**Runtime.** Under a minute.

**Steps.**

1. Setup: locate the article, load the configuration.
2. Run the stage: population and folds.
3. Read what it wrote: the population per dataset and the fold manifest.

## 1. Setup

Locate the article folder, import the paper code and load the configuration. Loading the configuration verifies the SHA-256 of every dataset, so a wrong or edited data file stops the run here. `CONFIG` is the one knob: point it at another YAML to run a variant into another folder.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """publication/2_journal_article, found from this folder, JupyterLab's root or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "paper" / "stages.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "paper" / "stages.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
from paper import config, results, stages  # noqa: E402

CONFIG = ARTICLE / "config" / "evaluation.yaml"   # point this at another configuration to run a variant
SETTINGS = config.load(CONFIG)                     # verifies the dataset hashes before anything runs
RESULTS = SETTINGS.output_root()
print("results folder:", RESULTS.relative_to(ARTICLE))

## 2. Population and folds

A site-day enters the evaluation only when all 96 fifteen-minute readings are present. Each of the 18 stations is held out once; anything fitted for a fold is fitted on the other Beta stations only (an Alpha station is scored with the fit on all eight Beta stations), so no station ever sees its own labels.

In [ ]:
out = stages.data_folds(SETTINGS)

## 3. What was written

`population.csv` counts the site-days per dataset, how many are complete, and how many complete headline-confidence days carry a labelled wrong sign. `fold_manifest.csv` lists, for every fold, the held-out station, the M8 training stations, the M9 calibration stations and a hash of the exact training keys.

In [ ]:
display(out['population'])
out['manifest'][['fold_id', 'cohort', 'held_out', 'm8_training_stations', 'm9_calibration_stations']].head(18)

## Result

The population and the folds are fixed for every method. The next notebook fits and scores the two previous methods on them.